# RR-SNN v2：从解析原语到真实事件、可训练策略与可复现实验

主候选在预先锁定的同 CPU 对照下落后；这轮结果不支持晋升该候选。

本报告保存完整结果、模型定义、方法边界和尚未闭合的外部验收条件。数学原语正确、合成任务有效、实际系统可部署分别有各自的证据要求。本轮没有将任何未注册研究状态激活到生产 Hepta，也没有生成官方 owner 回执。

主矩阵：14 组配置 × 5 seeds = 70 次训练。新 train/validation/test/shift 数据量分别为 1024/256/512/512；另有 128 个匹配价值组、每组四个 A/B×冲击反事实。全部主要模型含 977 个参数，共用 22 维观察与 81 动作空间。

真实脉冲反馈补充实验独立训练五个种子，每条轨迹保存固定更新和同 CPU 目标两个里程碑。其同 CPU 目标下的合成 shift 效用差为 **-10.76 [-13.41, -8.21] 个百分点**。该结果与主矩阵分别检验、分别记录；完整路径可训练性没有自动带来任务性能优势。

## 从原论文借用了什么

[He 与 Xiong (2012)](https://www.princeton.edu/~wxiong/papers/rollover.pdf) 把续约损失、继续价值和内生违约边界联立求解，使用平稳债务结构以及 Laplace 变换等解析技术。原文允许增发权益承担损失；本实验另加的硬资源预算和有限期限队列不是原文直接给出的神经元模型。

| 本项目保留的数学结构 | 神经算子中的明确选择 |
| --- | --- |
| 到达边界的事件 | 证据首达触发脉冲，随后 reset 与不应期 |
| 继续价值与更新成本相互作用 | 记忆续约、输出与释放共用资源账本 |
| 期限影响当前决策 | 保留逐项期限，比较相同均值下的不同到期分布 |
| 解析传播与边界条件 | 受限标量解析核；有限 CTMC 矩阵指数；真实事件采样 |

这是一个需要实验检验的工程建模选择；金融变量没有被直接当成生物变量，当前也没有生物神经记录拟合证据。

### 任务效用、内部脉冲与预算的单位

任务包含四个带价值和期限的记忆项。到期时可释放、续约两单位时间，或输出提前结果；续约／输出各耗费一单位预算，释放免费。终点 T=4 的左极限查询仍存活的项目。给定 early_penalty=0.15，总效用为

$$U=\sum_{i\,\mathrm{emitted}} v_i e^{-0.15(4-t_i)}+\sum_{i\,\mathrm{alive\ at}\ 4^-}v_i.$$

资源冲击先扣除当前可用预算，补充随后按固定事件时序入账；各模型共用精确合法动作 mask。因此预算不透支是共同环境约束的结果，不能单独归功于某个神经元。归一化效用是每个场景 U/U* 再取均值，U* 由独立动态规划求出。

这里有两个不同的事件单位：隐藏神经元的内部 spike，以及任务 head 选择的记忆输出动作。任务预算对后者及续约计费；内部 spike 未逐次从同一账本扣款。边界公式中的 cost 是由资源／期限信息构造的策略参数，不是已测量的每个脉冲能耗。实际计算成本由独立 CPU、RSS、吞吐与时延测试记录。

## 1. 已冻结主比较的结果

主候选是 `rr_density_sampled`：以观测端点状态轨迹的似然比梯度训练，部署评估由真实 Gillespie CTMC 执行。主对照 `ctrnn_matched_cpu` 在打开新 holdout 前锁定，每个 seed 的 CPU 训练预算目标等于候选对应 seed 的实测值；停止发生在完整 epoch／validation 边界，实际超出量保留在 receipt，不宣称逐秒相等。普通固定预算组均为 32 epochs、256 次 Adam 更新。

| 数据 | 归一化效用差 / 百分点 [95% CI] | 负迁移 | 正迁移 | 实际配对回放数 |
| --- | --- | --- | --- | --- |
| test | -29.12 [-31.96, -26.31] | 70.94% | 3.87% | 2560 |
| shift | -11.93 [-14.90, -9.11] | 40.82% | 16.95% | 2560 |
| paired | -22.21 [-24.12, -20.23] | 66.58% | 2.80% | 10240 |

差值方向为候选减去对照。负迁移按每次真实回放的原始效用比较，在平均回放之前取符号。置信区间对训练 seed 与共享场景／价值组做成组配对 bootstrap，10,000 次；paired 的四个条件保持在同一价值组内。区间条件于已固定的有限路径回放，并不额外估计无限未来脉冲采样的全部方差。只有五个训练 seed，结论的外推范围有限。

### 全部主配置

| 配置 | 验证 NLL | test 效用 % | shift 效用 % | 训练 CPU 秒均值 | 更新数均值 |
| --- | --- | --- | --- | --- | --- |
| RR：采样训练、真实 CTMC | 1.35 ± 0.05 | 65.92 ± 1.65 | 55.56 ± 2.98 | 532.8 | 256 |
| 多期限／固定边界 | 1.40 ± 0.08 | 65.20 ± 1.27 | 54.55 ± 2.24 | 417.9 | 256 |
| FP／无续约分支 | 1.32 ± 0.05 | 66.24 ± 1.82 | 56.15 ± 2.63 | 418.1 | 256 |
| RR：期限分支取均值 | 1.34 ± 0.04 | 66.80 ± 2.12 | 56.25 ± 1.98 | 418.0 | 256 |
| RR：期限分支固定重排 | 1.32 ± 0.04 | 67.28 ± 2.01 | 57.85 ± 2.41 | 411.6 | 256 |
| RR：冻结资源分支 | 1.34 ± 0.05 | 68.04 ± 2.34 | 57.52 ± 3.03 | 378.1 | 256 |
| RR：分布期望 | 1.13 ± 0.02 | 81.54 ± 1.47 | 64.46 ± 2.75 | 370.5 | 256 |
| RR：均值闭合消融 | 1.09 ± 0.02 | 83.18 ± 0.77 | 64.45 ± 2.42 | 401.0 | 256 |
| 解析区间 LIF | 1.10 ± 0.02 | 81.79 ± 1.31 | 62.93 ± 3.73 | 12.2 | 256 |
| 事件刷新 ALIF | 1.09 ± 0.01 | 82.26 ± 0.65 | 64.72 ± 3.68 | 13.9 | 256 |
| 资源 LIF | 1.10 ± 0.02 | 82.05 ± 1.71 | 64.10 ± 3.73 | 9.5 | 256 |
| 连续时间 RNN | 1.10 ± 0.01 | 80.68 ± 1.40 | 63.43 ± 3.36 | 5.0 | 256 |
| 状态预算匹配 RNN | 1.10 ± 0.01 | 82.03 ± 0.76 | 62.71 ± 2.09 | 5.1 | 256 |
| CPU 预算匹配 RNN | 0.47 ± 0.06 | 95.04 ± 1.18 | 67.50 ± 1.49 | 533.0 | 20859 |

“±”是五个 seed 的样本标准差，不是置信区间。所有选择使用 validation，不按 test/shift 的最好结果选择基线。次要对照由预列候选中的最低平均 validation NLL 确定：`alif`。零 oracle 效用的场景统一记归一化效用为 1；两模型在这些场景的差为零，原始效用与零分母场景数量均保存在原始 JSON。

In [ ]:
# 主结果：五个训练 seed 的均值与标准差；这是合成数据比较。

## 2. 相同当前状态、不同期限结构

每个组共用四个记忆价值、初始预算 4、记忆数 4、平均期限 2；A 的期限为 `(1,1,3,3)`，B 为 `(2,2,2,2)`。两者初始输入仅第 7–10 号期限特征不同。预算冲击为 0 或 2，发生在 t=1.5；t=2.5 补充 1。价值与未来资源日程在 t=0 对所有模型同样宣布，后续价值 cue 不重复给出。这里测量的是对已宣布未来压力的提前调整。

每个随机策略、每个 seed 回放四次，确定策略一次并在比较中广播；没有把广播复制当作新增独立样本。终末保留值、高值项目保留比例及同数量保留超额同时记录，但这些指标不包含提前输出收益，不能取代总效用。

In [ ]:
# 匹配反事实：A/B 有相同平均期限；误差条为 seed SD。

### 补充：共同的实际初始神经状态

主矩阵匹配了初态分布，四条件的实际初态抽样各自独立。另一个在打开 holdout 前声明的诊断，先从正确 reset 分布为每个价值组抽一个完整实际状态，再复制给四个条件。初始 indices、电位、context、drift、boundary、time、counts 与 observed 标记逐位相同；该主模型没有独立 spike-trace buffer。四份复制只计为一个初态样本。四个审计 seed 各 4096 组的边际检验最大 |z|=1.773。

共同状态指 t=0 观察之前；期限介入后控制与后续路径可以分化，未来噪声没有强制共享。该实验仍使用同一因果账本与对照，5 个训练 seed、128 组、4 次真实回放得到：

| 指标 | 严格共同初态诊断 |
| --- | --- |
| 候选归一化效用 % | 74.13 ± 1.44 |
| 同 CPU RNN 效用 % | 96.52 ± 1.23 |
| 效用差 / 百分点 [95% CI] | -22.39 [-24.13, -20.62] |
| 负迁移比例 | 66.85% |

| 终点诊断 | 主 RR 候选 | 同 CPU RNN |
| --- | --- | --- |
| 保留价值 | 2.1905 ± 0.0248 | 2.4702 ± 0.0460 |
| 最高两项的保留比例 % | 62.22 ± 0.78 | 80.18 ± 2.51 |
| 相同保留数量的价值超额 | -0.0022 ± 0.0082 | 0.2203 ± 0.0220 |

终点指标不包括提前输出的收益，须与总效用同时解读。原始主 paired 结果未被替换。证据位于 `audit/strict_initial_counterfactual/`。

### 提前动作的分布检查

独立 probe 在 A 场景 t=1 的首个决策比较宣布冲击 0 与 2。此刻预算、alive、价值和当前到期状态完全相同；冲击尚未发生。随机候选对每个价值组与条件使用 64 条真实脉冲路径，报告期望续约数与输出数的处理效应。随机单路径动作不一致本身不能证明提前响应；经验动作分布 TV 在有限样本下上偏，因此只作描述性指标。

完整 probe 和处理效应区间位于 `results/anticipation_probe.json` 与主 aggregate 的 `anticipation_probe` 字段。

| 策略 | 冲击 2−0：续约数 [95% CI] | 冲击 2−0：输出数 [95% CI] |
| --- | --- | --- |
| CPU 预算匹配 RNN | -0.99 [-1.09, -0.91] | +0.76 [+0.56, +0.95] |
| RR：分布期望 | -0.94 [-1.46, -0.48] | +0.94 [+0.48, +1.46] |
| RR：采样训练、真实 CTMC | -0.05 [-0.10, -0.01] | +0.05 [+0.01, +0.10] |

这些区间比较同一策略对宣布冲击的动作响应；动作响应本身不等于比其他策略获得更高效用。

## 3. 修复的动力学与仍存在的模型近似

连续标量原语采用到阈值的距离 $X_t=x+μ t+σ W_t$，$x>0$，在 0 吸收。其首达密度与存活联合密度分别为

$$f(t\mid x)=\frac{x}{\sigma\sqrt{2\pi t^3}}\exp\!\left[-\frac{(x+\mu t)^2}{2\sigma^2t}\right],$$

$$K_t(y\mid x)=\frac{\exp[-(y-x-\mu t)^2/(2\sigma^2t)]}{\sigma\sqrt{2\pi t}}\left(1-\exp[-2xy/(\sigma^2t)]\right),\quad y>0.$$

这里的 K 是未归一化的 killed kernel，积分为存活概率；条件于未发放时才除以该积分。首达事件使用 f，末段没有事件时使用存活概率；折现首达量不能直接当作发放概率。输入事件之间的生存传播可定义为 $\mathcal A_tg(x)=e^{-\rho t}\int_0^\infty K_t(y\mid x)g(y)dy$，再连接首达输出、reset 与不应期的 renewal 算子。以上是经典首达数学的应用，不声称它们为本项目新定理。

v2 保存完整有限状态概率分布，而非把条件均值重新当作单点证据。默认每个神经元有 16 个活动电位节点、2 个 Erlang 不应期相位。跨阈值跳转产生脉冲，随后经过不应期相位并重置；同一区间可以反复发放。行向量约定为

$$p(t+\Delta)=p(t)\exp(Q\Delta).$$

累计期望计数通过带 reward column 的增广矩阵指数计算。独立长区间测试得到 8.733 次期望发放，未施加每段一次的上限。步进顺序为：用旧控制推进到当前时间、接收当前观察、更新控制与跳变、读出。首帧 dt=0；no-op 只推进时间，不能偷偷重算控制。

独立审计中，有限 CTMC 的半群误差为 4.44×10⁻¹⁵；32,768 条真实 Gillespie 路径的平均发放计数对矩阵指数参照相差 0.740 个标准误。这些是指定有限模型的实现证据。相对于连续 Brownian＋确定性不应期参照，默认 N=16、K=2 在一组明确测试中仍有 +0.088614 的平均计数误差。空间网格、反射下界、相位数和观测 remap 分开检查，不能由极小的矩阵求值误差推断连续模型误差同样小。

In [ ]:
# 三个独立收敛方向。每个面板固定其余条件；原始参数与参照值在审计 JSON 中。

### 真实事件方法与数值替代

LIF 对照使用恒定电流区间的解析首达时间、发放周期、reset、不应期以及所有事件时刻的指数迹，未使用小步 Euler 作为唯一参照。三个脉冲基线各 200 个随机区间与独立标量事件循环核对，整数计数差为零，状态最大差约 4.05×10⁻¹⁵。ALIF 是观察事件刷新、区间内保持阈值的明确版本；它没有完整复现连续衰减阈值 ALIF／LSNN 的训练方案。

这些 LIF 对照的 hard forward 执行整数真实事件；训练 backward 使用 sigmoid／连续计数 surrogate，属于有偏梯度估计。资源 LIF 用当前观测预算调节阈值，也是启发式策略，不宣称求解完整资源控制问题。

同一权重下，matrix exponential 与 uniformization 的验证集实现控制已覆盖五个 seed；另有数值求根边界＋uniformization 的完整有限算子控制。完整控制中，36 个标量根对独立 brentq 的最大差为 4.88×10⁻¹⁵，隐函数梯度有限差分误差 3.28×10⁻¹⁰。五个正式权重、每个 64 个 validation 场景共 570 个合法决策没有 argmax 差异；logits 最大差 1.91×10⁻⁵，expected-set NLL 最大差 4.77×10⁻⁷，全部参数梯度相对 L2 差最大 3.26×10⁻⁶。此处 backward 专指这些采样训练权重上的确定性 expected-distribution NLL，不包括 endpoint likelihood-ratio 或 coupled full-path 目标的数值替代比较。

数值控制不改变资源任务和读出，也不把有限 CTMC 当成已求出的连续 PDE。该控制比较求值与局部反向梯度，不声称进行了两套实现的全流程重新训练。

### 相同权重、不同传播／决策方式

| 训练版本 | 实际回放模型 | 数据 | 归一化效用 % | 相对原执行方式差 / 百分点 [95% CI] |
| --- | --- | --- | --- | --- |
| RR：采样训练、真实 CTMC | brownian | test | 66.55 ± 2.94 | +0.63 [-0.76, +2.26] |
| RR：采样训练、真实 CTMC | brownian | shift | 56.43 ± 3.63 | +0.86 [-0.58, +2.46] |
| RR：分布期望 | brownian | test | 82.19 ± 0.64 | +0.65 [-0.61, +2.04] |
| RR：分布期望 | brownian | shift | 65.83 ± 2.61 | +1.37 [+0.09, +2.79] |
| RR：分布期望 | ctmc | test | 80.79 ± 1.28 | -0.75 [-1.79, +0.31] |
| RR：分布期望 | ctmc | shift | 65.39 ± 3.00 | +0.94 [-0.49, +2.41] |

Brownian 回放采用连续证据、确定性不应期和真实首达；它与有反射下界／Erlang 相位的有限 CTMC 是不同物理模型。将分布训练权重用于真实 CTMC 也会改变非线性动作决策，因此这些是事先声明的迁移诊断，不是相同模型精确相等的检查。没有根据该表重新选取主候选。

## 4. 三种训练证据的清楚区分

| 层次 | 随机对象与梯度 | 能够支持的结论 |
| --- | --- | --- |
| 标量删失校准 | 首达密度及末段 survival | 受限模型的事件概率、删失似然与参数拟合 |
| 主候选 endpoint 训练 | 观测端点 CTMC 状态；区间内事件积分消去 | 观测状态轨迹的期望决策 NLL 梯度 |
| 真实事件路径训练 | 实际跳变计数、驻留时间、初态／remap 概率 | 真实事件网络在固定示教输入上的期望决策 NLL 梯度 |

主候选的 score-function 估计同时保留损失的直接导数：

$$\nabla_\theta E[L]=E[\nabla_\theta L+(L-b)\nabla_\theta\log p_\theta(Z)].$$

baseline 使用其他序列的 leave-one-sequence-out 损失并 detach；条件于输入与参数，各序列采样独立。独立小模型完全枚举了 96 条非零端点路径，损失与四个参数梯度的 Monte Carlo 结果与精确枚举相符。这类训练目标是固定示教输入的决策 NLL，并非动作改变资源环境后的终点效用策略梯度。

另外完成两神经元真实首达、reset、确定不应期的延迟 count 反馈训练：三个 seed 的精确期望 cost 从 1.951156 降到约 0.636，路径 score 梯度有独立 renewal-count 分布的参照。这是指定小网络的训练证据。

### 补充：真正由脉冲反馈驱动的循环网络

主矩阵的 CTMC 控制器由确定性观察历史 context 驱动。为覆盖随机脉冲反馈，本轮另实现 `coupled/`：循环权重只读取真实脉冲的指数迹，head 只读取脉冲迹，控制只在真正观察事件更新。当前 remap 即时脉冲立即进入 head，在下一次真正观察才更新循环控制。

每段真实 Gillespie 路径的对数似然为

$$\log p=\sum_{i\ne j}N_{ij}\log q_{ij}-\sum_i T_i\lambda_i+\log p_{\mathrm{initial}}+\sum\log P_{\mathrm{remap}}.$$

它包含最后的右删失驻留时间和参数相关 reset 分布。漏掉初态／重复 reset 的边界导数会产生明显错误。补充计划在打开 holdout 前冻结，分别保存固定 256 更新与同 CPU 总预算的 validation 最优检查点；这是同一训练轨迹的两个里程碑。结果单独保存在 `coupled/results/aggregate.json`，不替换主比较，也不称为连续时刻即时反馈的全异步网络。

### 脉冲反馈网络的独立结果

| 预算 | 数据 | 预定对照 | coupled 效用 % | 对照效用 % | 差 / 百分点 [95% CI] | 负迁移 |
| --- | --- | --- | --- | --- | --- | --- |
| 固定 256 更新 | test | 状态预算匹配 RNN | 64.50 ± 0.57 | 82.03 ± 0.76 | -17.53 [-20.24, -14.97] | 48.20% |
| 固定 256 更新 | shift | 状态预算匹配 RNN | 52.04 ± 0.80 | 62.71 ± 2.09 | -10.67 [-13.38, -8.12] | 41.25% |
| 固定 256 更新 | paired | 状态预算匹配 RNN | 73.16 ± 0.41 | 86.73 ± 0.29 | -13.57 [-14.34, -12.87] | 53.90% |
| CPU 预算目标匹配 | test | CPU 预算匹配 RNN | 70.60 ± 2.49 | 95.04 ± 1.18 | -24.44 [-27.18, -21.76] | 65.86% |
| CPU 预算目标匹配 | shift | CPU 预算匹配 RNN | 56.74 ± 1.51 | 67.50 ± 1.49 | -10.76 [-13.41, -8.21] | 41.13% |
| CPU 预算目标匹配 | paired | CPU 预算匹配 RNN | 76.89 ± 1.99 | 96.52 ± 1.23 | -19.63 [-21.43, -18.04] | 64.79% |

| 预算 | 验证 NLL | 训练 CPU 秒 | 更新数 | 实际训练 spikes | CPU 目标超额秒 |
| --- | --- | --- | --- | --- | --- |
| 固定更新 | 1.54 ± 0.05 | 139.32 ± 35.09 | 256.00 ± 0.00 | 170894.80 ± 35134.23 | 0.00 ± 0.00 |
| 同 CPU 目标 | 1.25 ± 0.04 | 536.13 ± 15.22 | 1051.20 ± 214.18 | 1063076.20 ± 260721.22 | 3.37 ± 1.29 |

两行来自五条训练轨迹的不同里程碑，不当作十个独立训练 seed。固定更新行不按 CPU 预算停止，超额栏记为零。

In [ ]:
# 真实脉冲反馈网络：两个事先指定的预算比较；全部数据为合成任务。

## 5. 期限结构必要性的严格有限反例

在另一个完全规定的有限控制模型中，Y=2、budget=2、两个义务价值均为 1、记忆总量 2、平均期限 2，仅改变期限 A=(1,3)、B=(2,2)，得到：

| 首动作 | A 的 Q 值 | B 的 Q 值 |
| --- | --- | --- |
| 等待 | 3.9403969893 | 2.1843707397 |
| 最佳立即输出 | 3.6648448011 | 2.5875900237 |
| 严格最优 | 等待 | 立即输出 |

若 A/B 各有一半先验概率，首步只看到合并汇总状态，即使首步之后恢复完整期限信息，最小平均 regret 仍为 0.1377760941。随机化首动作不能超过线性目标的最佳纯动作。这个构造证明仅保留均值等汇总量并不普遍充分；它不证明 RR 网络胜过同样获得期限信息的 RNN。完整 specification、569 个已求值状态、初始 Q 表与前向回报／预算核验保存在 `audit/results/full_control_bellman.json`。

## 6. Lambert-W 特例与完整控制问题

对向上证据 $dY=a\,dt+\sigma\,dW$、无限时域、固定输出效用 Y−c、reset 为 $y_r$、固定不应期，假设 $c>y_r$、$\rho,\sigma>0$、$\tau\geq0$，且不应期内无其他状态变化。对这个标量问题，值匹配和光滑拼接给出

$$\gamma=\frac{\sqrt{a^2+2\rho\sigma^2}-a}{\sigma^2},\qquad h=c+\frac{1+W_0(-\exp[-\rho\tau-1-\gamma(c-y_r)])}{\gamma}.$$

这条 Lambert-W 表达式由上述限定神经控制问题推导，不是 He–Xiong 原文的违约边界公式。该特例的 QVI 残差与阈值策略最优性已经核验。当前控制器把资源和期限压力映射为 cost，再使用该公式并裁剪边界，是一个具有解析结构的策略类。它没有求解整个资源、期限与延迟网络的价值函数。

完整状态至少包含证据、预算、逐项剩余期限、价值、不应期、已宣布外生事件与未到达的延迟事件。内部 QVI 需要等待生成元、输出／释放干预算子，并在到期与资源冲击处指定联合边界跳转；共享预算使逐项独立最大化通常失效。不应期内资源和期限继续变化时，继续价值必须使用完整转移算子，不能只乘一个折现因子然后 reset。详细推导与合法适用范围在 `audit/full_control_qvi.txt`。本轮未求出一般多维连续 QVI 的全局解析解或普遍策略误差界。

### 续约压力映射到 cost 的含义需要保留

在上述冻结标量问题中，令 $q=\exp[-\rho\tau-\gamma(h-y_r)]$，由隐函数求导得到 $\partial h/\partial c=1/(1-q)>0$。因此，其他条件不变时，提高输出成本会推迟向上的首达输出。当前实现把期限压力的一部分加到这个 cost 上，是一种具体策略归纳偏置。它与把将来到期的续约负担直接写入继续价值、从而改变提前输出意愿的完整控制问题并不相同。

这说明“金融模型能给出内生边界”还不足以证明当前神经元的压力映射方向最优。本轮消融和任务结果用于检验这个选择；不能把正确的求根或梯度检查当作该建模选择已被证实。

## 7. 资源成本与 Hepta 对接

资源实验的横轴是 64/256/1024/4096 个同时驻留的策略状态，共享一组权重，每个策略含八个隐藏单元。这些是本机参考实现的策略实例；没有把它们改称真实生产 Hepta Cell。

主测量固定 CPU 7、单线程、5 条预热序列、25 条八步序列，共 200 个全行批处理 tick 时延样本。主表报告 tick p50/p95/p99、row-ticks/s 和峰值进程 RSS；25 个 whole-sequence 样本另列，其 p99 仅作探索。所有实例状态先驻留，再按同一 microbatch 协议处理。

训练中的首轮成本测试发现共享 cgroup 内存回收压力；原始极端值完整保留在 `benchmark_results/memory_pressure_run/`，未据此形成稳定算法尾延迟结论。主表由独立 observer 记录每行测量前后的 memory.events/current/stat 与磁盘余量，发生触顶、OOM 或状态未知的行不能被静默当作正常结果。

全部主要模型的可训练参数均为 977；676 B 是共同的数值状态上限，不是每种模型都占用完全相同字节。实际推理持久数值状态为：普通 RNN 64 B、LIF 128 B、资源 LIF 160 B、ALIF 192 B、真实 CTMC 233 B、coupled 305 B、分布版／匹配 RNN 672 B、冻结资源分布版 676 B。主候选训练时使用完整端点分布，部署时真实 CTMC wrapper 去除冗余概率表，不能把部署 233 B 当作其训练占用。固定 buffers、RNG、Python 对象、临时 Q、反向图及进程基础开销另计。资源账本是合成任务的无量纲预算，没有把脉冲次数换算成未经测量的焦耳。

### 完整本机成本表

| 实现 | 实例数 | p50 ms | p95 ms | p99 ms | 实例-ticks/s | 进程峰值 MiB | 资源窗口 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| RR：采样训练、真实 CTMC | 64 | 13.852 | 21.628 | 26.016 | 4285.2 | 257.7 | 未观察到限额事件 |
| RR：采样训练、真实 CTMC | 256 | 53.656 | 150.077 | 451.794 | 3607.8 | 313.3 | 未观察到限额事件 |
| RR：采样训练、真实 CTMC | 1024 | 173.603 | 225.802 | 280.043 | 5760.2 | 328.5 | 未观察到限额事件 |
| RR：采样训练、真实 CTMC | 4096 | 631.221 | 1165.247 | 1695.492 | 5856.5 | 326.9 | 未观察到限额事件 |
| 解析区间 LIF | 64 | 0.519 | 1.096 | 1.591 | 108074.6 | 208.9 | 未观察到限额事件 |
| 解析区间 LIF | 256 | 0.671 | 1.127 | 1.603 | 354413.4 | 209.5 | 未观察到限额事件 |
| 解析区间 LIF | 1024 | 1.413 | 2.345 | 3.042 | 671036.4 | 213.6 | 未观察到限额事件 |
| 解析区间 LIF | 4096 | 4.154 | 8.205 | 12.413 | 885447.6 | 231.3 | 未观察到限额事件 |
| 连续时间 RNN | 64 | 0.087 | 0.300 | 0.609 | 493466.1 | 207.8 | 未观察到限额事件 |
| 连续时间 RNN | 256 | 0.193 | 0.482 | 0.669 | 1098637.5 | 208.0 | 未观察到限额事件 |
| 连续时间 RNN | 1024 | 0.467 | 0.874 | 1.238 | 1978152.2 | 211.6 | 未观察到限额事件 |
| 连续时间 RNN | 4096 | 1.570 | 2.817 | 3.185 | 2394140.0 | 226.1 | 未观察到限额事件 |
| 状态预算匹配 RNN | 64 | 0.105 | 0.267 | 0.324 | 463524.3 | 207.8 | 未观察到限额事件 |
| 状态预算匹配 RNN | 256 | 0.371 | 1.094 | 1.582 | 526788.1 | 208.2 | 未观察到限额事件 |
| 状态预算匹配 RNN | 1024 | 1.276 | 2.636 | 3.015 | 692480.5 | 213.2 | 未观察到限额事件 |
| 状态预算匹配 RNN | 4096 | 3.770 | 9.540 | 11.402 | 895272.8 | 238.8 | 未观察到限额事件 |
| CPU 预算匹配 RNN | 64 | 0.121 | 0.334 | 0.636 | 369897.1 | 207.8 | 未观察到限额事件 |
| CPU 预算匹配 RNN | 256 | 0.360 | 0.921 | 1.526 | 573759.9 | 208.3 | 未观察到限额事件 |
| CPU 预算匹配 RNN | 1024 | 1.091 | 2.545 | 3.248 | 801582.2 | 214.2 | 未观察到限额事件 |
| CPU 预算匹配 RNN | 4096 | 3.725 | 9.286 | 10.827 | 919665.4 | 238.9 | 未观察到限额事件 |
| RR：真实脉冲反馈 | 64 | 16.668 | 23.974 | 39.492 | 3571.4 | 233.6 | 未观察到限额事件 |
| RR：真实脉冲反馈 | 256 | 58.468 | 91.314 | 198.519 | 3806.1 | 277.2 | 未观察到限额事件 |
| RR：真实脉冲反馈 | 1024 | 264.525 | 573.921 | 840.745 | 3219.1 | 288.7 | 未观察到限额事件 |
| RR：真实脉冲反馈 | 4096 | 1008.319 | 2135.225 | 3219.207 | 3202.7 | 309.7 | 未观察到限额事件 |

每个 tick 处理该行全部 N 个策略实例。吞吐单位是实例-ticks/s，不能把批次时延除以 N 后当作逐请求 p99。进程 RSS 包含 Python、框架、输入与输出、工作区和分配器；它不等于状态数组大小。这批共享权重的状态实例也不包含真实 Hepta owner 路由和 Cell 组件成本。

coupled 使用事先固定的 equal_cpu seed101 权重，按相同输入和计时协议单独增补。其原生推理仍计算跳变计数、驻留时间和路径 score；这些当前可优化的实现开销全部计入，不能据此声称架构理论下界。所有受压或资源状态未知的行原样保留并标记，不形成未经限定的稳定尾延迟结论。

### 已实际完成的 Hepta 离线集成

| 项目 | 证据 |
| --- | --- |
| 真实源码编译 | 锁定提交 f8c9419df6ddeb36ff8b522a50219981fb8d9345；三个完整原始 crate 的库目标在缩小工作区编译，62 个源文件字节未改 |
| 官方类型与摘要 | 5 项适配合约测试通过；codec、feature/tick digest 与独立 Python 重算一致 |
| Rust 物理 checkpoint | 6 项测试通过；完整 RNG 与状态两条 Rust 往返路线重现后续三个真实脉冲 |
| 正式权重联合恢复 | 四个恢复点重现实际脉冲、动作、期限队列、资源账本与终态；八类非法恢复拒绝 |
| 新脉冲反馈状态恢复 | 另建完整 trace/context/pending-score/RNG 适配器；两个 seed101 里程碑共八个恢复点、十八个拒绝案例通过 |
| 审阅 shadow proposal | 真实八维 sampled latent 经过官方类型适配，Q24 最大误差 2.98×10⁻⁸；owner_executed=false，activation=false |

上述构建不等同于整个 monorepo 编译或正式 owner 执行。研究 envelope 尚未注册到生产协议；原 V1 对未知物理状态字段的拒绝行为已保留。

## 8. 哪些 gate 尚未闭合

当前可调用环境没有验证过的 dot／目标机执行入口，没有正式 inference owner 回执，也没有真实独立未来窗口与真实 Cell 工作负载。因此无法完成目标机上的 64/256/1024/4096 Cell 资源验收、真实未来效用与负迁移验收，或证明生产 journal／witness／CAS 的恢复行为。这些缺失不能用合成 shift、独立状态复制或只读源码访问替代。

要继续实际 R5 评估，需要接入明确的目标执行环境、实际 owner 接口与身份、冻结的未来窗口数据及评价目标，以及 Cell 的真实工作负载和成本口径。研究包已经提供可执行离线入口与审阅 proposal；尚未获得的回执字段明确保持为空。完整逐项状态见总 `closure_ledger.json`、原 `hepta/closure_ledger.json` 及 `audit/coupled_checkpoint/closure_addendum.json`；主任务统计结论与外部集成证据分别记录。

## 9. 新颖性与结论的边界

原论文的可迁移内容是内生边界、继续价值与续约损失相互作用的建模方法。论文采用平稳、均匀错开的债务期限结构；本任务的有限非平稳期限队列需要另外定义状态和控制问题。首达神经元、精确事件模拟、自适应阈值、连续时间解析网络与多时间尺度记忆已有公开研究。因此不能把任意一种已有算子加上金融名称就宣称原创。可检验的贡献应是具体的期限／资源耦合机制、正确的事件与梯度实现、可证明的信息必要性，以及公平任务中的实际收益。

本轮的主要新实证是：旧闭合问题得到有限模型内的修复；真实脉冲反馈与完整路径梯度可执行；期限汇总存在严格反例；候选优劣由已冻结实验决定。没有生物神经数据拟合、没有证明新的生物机制，也没有建立文献优先权或生产收益。

机制消融只改专用分支，原始 22 维观察仍相同。固定重排不是删除期限信息，均值分支也不使公共输入完全看不到逐项期限。均值闭合消融只能与分布期望版比较，不能为它强行构造同律单路径。主 v2 数据与架构均变更，不能把 v0/v2 的分数差单独归因为一个修复。

## 10. 原始文献与复现

- [Rollover Risk and Credit Risk](https://www.princeton.edu/~wxiong/papers/rollover.pdf) (2012)：Primary source: stationary uniformly staggered debt, rollover losses and endogenous default. Its closed form does not automatically cover arbitrary finite deadline queues.
- [Exact simulation of integrate-and-fire models with exponential currents](https://romainbrette.fr/exact-simulation-of-integrate-and-fire-models-with-exponential-currents/) (2007)：Primary author source for pre-existing exact event simulation methods; accuracy/speed controls should include exact event methods.
- [Long short-term memory and Learning-to-learn in networks of spiking neurons](https://papers.nips.cc/paper_files/paper/2018/hash/c203d8a151612acf12457e4d67635a95-Abstract.html) (2018)：Primary conference source for adaptive spiking neurons and LSNNs. The implemented event-held ALIF comparator is narrower than the complete LSNN training setup.
- [Closed-form continuous-time neural networks](https://cap.csail.mit.edu/sites/default/files/research-pdfs/Closed-form%20continuous-time%20neural%20networks.pdf) (2022)：Primary institutional paper showing closed-form continuous-time neural architectures pre-exist this project. Our comparator is explicitly a time-basis CTRNN, not a claimed CfC reproduction.
- [Legendre Memory Units: Continuous-Time Representation in Recurrent Neural Networks](https://papers.nips.cc/paper_files/paper/2019/hash/952285b9b7e7a1be5aa7849f32ffff05-Abstract.html) (2019)：Primary conference source for mathematically constructed temporal memory bases, including a spiking implementation; multi-timescale memory alone is not a novelty claim.
- [Spiking Neurons and the First Passage Problem](https://digitalcommons.rockefeller.edu/knight_laboratory/10/) (2011)：Primary institutional source for first-passage models of interspike intervals, predating the proposed financial-model transfer.

```bash
python -m rrsnn_v2.density_validation
python -m rrsnn_v2.validate_spiking_baselines
python -m rrsnn_v2.run_matrix --workers 6
python -m rrsnn_v2.run_evaluations --workers 6
python -m rrsnn_v2.anticipation_probe
python -m rrsnn_v2.aggregate_results
python rrsnn_v2/hepta/run_offline_replay.py --checkpoint rrsnn_v2/results/rr_density_sampled/seed_101/model.pt --joint-output-name rr_density_sampled_seed101
```

训练与全部评估结束后，资源表使用 `python rrsnn_v2/benchmark_resource_observer.py run`。补充循环 SNN 的命令与冻结计划在 `coupled/`。训练、参数、数据和评估源码的 SHA 均写入 receipt；聚合在文件不齐或哈希漂移时拒绝输出部分主表。原始 logits／动作与逐次脉冲记录、全部模型检查点、独立审计和 vendored Rust 依赖保存在研究包中。